In [10]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont, ImageEnhance

OUT_DIR = Path("media-site/animations/white_dwarf")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Подставь свой файл, если имя другое
BACKGROUND_PATH = Path("media-site/assets/WD3.jpg")

OUT = OUT_DIR / "white_dwarf_seismology_live.gif"

FPS = 24
FRAMES = 160

CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
ORANGE = (255, 160, 70)
RED = (255, 80, 110)
WHITE = (245, 250, 255)

rng = np.random.default_rng(26051)


def load_font(size):
    for path in [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()


FONT_SMALL = load_font(22)
FONT_TINY = load_font(16)


bg0 = Image.open(BACKGROUND_PATH).convert("RGBA")
W, H = bg0.size

# немного приглушаем исходник, чтобы живые слои читались
bg0 = ImageEnhance.Brightness(bg0).enhance(0.82)
bg0 = ImageEnhance.Contrast(bg0).enhance(1.08)

# Координаты под сгенерированный 1792x1024 растр.
# Если используешь другой размер, всё масштабируется автоматически.
SX = W / 1792
SY = H / 1024

def sx(x): return x * SX
def sy(y): return y * SY

STAR_CX = sx(500)
STAR_CY = sy(440)
STAR_R = sx(314)

AMP_BOX = [sx(1400), sy(380), sx(1715), sy(560)]
RES_BOX = [sx(1040), sy(600), sx(1405), sy(710)]
BAR_BOX = [sx(1240), sy(880), sx(1590), sy(905)]
LIVE_DOT = (sx(1572), sy(34))
TIME_POS = (sx(454), sy(833))


def make_star_texture(size=720):
    tex = Image.new("RGBA", (size, size), (0, 0, 0, 0))
    d = ImageDraw.Draw(tex)

    cx = cy = size / 2
    r = size * 0.47

    # базовый диск
    for rr in np.linspace(r, 2, 80):
        a = int(8 + 105 * (1 - rr / r) ** 1.7)
        d.ellipse([cx - rr, cy - rr, cx + rr, cy + rr], fill=(*WHITE, a))

    # волокна/трещины
    for _ in range(420):
        a0 = rng.uniform(0, 2 * np.pi)
        rr = rng.uniform(0.05, 0.95) * r
        x = cx + np.cos(a0) * rr
        y = cy + np.sin(a0) * rr

        length = rng.uniform(18, 75)
        a1 = a0 + rng.normal(0, 0.9)

        x2 = x + np.cos(a1) * length
        y2 = y + np.sin(a1) * length

        if (x2 - cx) ** 2 + (y2 - cy) ** 2 < r ** 2:
            alpha = int(rng.uniform(25, 95))
            d.line([x, y, x2, y2], fill=(*CYAN2, alpha), width=1)

    # яркие узлы
    for _ in range(42):
        a0 = rng.uniform(0, 2 * np.pi)
        rr = rng.uniform(0.05, 0.92) * r
        x = cx + np.cos(a0) * rr
        y = cy + np.sin(a0) * rr
        rad = rng.uniform(3, 10)
        d.ellipse([x - rad, y - rad, x + rad, y + rad], fill=(*WHITE, int(rng.uniform(120, 230))))

    mask = Image.new("L", (size, size), 0)
    md = ImageDraw.Draw(mask)
    md.ellipse([cx - r, cy - r, cx + r, cy + r], fill=255)

    tex.putalpha(mask)
    return tex


STAR_TEX = make_star_texture()


def crop_panel(base, box):
    x0, y0, x1, y1 = map(int, box)
    patch = base.crop((x0, y0, x1, y1)).convert("RGBA")
    patch = ImageEnhance.Brightness(patch).enhance(0.55)
    return patch


AMP_BG = crop_panel(bg0, AMP_BOX)
RES_BG = crop_panel(bg0, RES_BOX)


def draw_live_star(img, phase):
    layer = Image.new("RGBA", (W, H), (0, 0, 0, 0))

    rot = STAR_TEX.rotate(
        phase * 360 / (2 * np.pi) * 0.18,
        resample=Image.Resampling.BICUBIC,
    )

    size = int(STAR_R * 2.08)
    rot = rot.resize((size, size), Image.Resampling.LANCZOS)

    x = int(STAR_CX - size / 2)
    y = int(STAR_CY - size / 2)

    pulse = 0.82 + 0.18 * np.sin(phase * 2.4)

    alpha = rot.getchannel("A").point(lambda p: int(p * 0.42 * pulse))
    rot.putalpha(alpha)

    layer.alpha_composite(rot, (x, y))

    d = ImageDraw.Draw(layer)

    # seismic rings
    for k in range(5):
        rr = STAR_R * (0.28 + ((phase * 0.12 + k * 0.19) % 1.0))
        a = int(90 * (1 - rr / STAR_R))
        d.ellipse(
            [STAR_CX - rr, STAR_CY - rr, STAR_CX + rr, STAR_CY + rr],
            outline=(*CYAN2, max(0, a)),
            width=max(1, int(2 * SX)),
        )

    # rim pulse
    rim_a = int(115 + 80 * np.sin(phase * 5) ** 2)
    d.ellipse(
        [STAR_CX - STAR_R, STAR_CY - STAR_R, STAR_CX + STAR_R, STAR_CY + STAR_R],
        outline=(*WHITE, rim_a),
        width=max(2, int(3 * SX)),
    )

    glow = layer.filter(ImageFilter.GaussianBlur(5))
    img.alpha_composite(glow)
    img.alpha_composite(layer)


def draw_panel_graph(img, box, bg_patch, phase, mode="amp"):
    x0, y0, x1, y1 = map(int, box)
    img.alpha_composite(bg_patch, (x0, y0))

    d = ImageDraw.Draw(img)

    w = x1 - x0
    h = y1 - y0

    # сетка поверх затемнения
    for gx in range(x0 + 20, x1 - 10, max(1, int(w / 6))):
        d.line((gx, y0 + 14, gx, y1 - 14), fill=(*CYAN, 45), width=1)

    for gy in range(y0 + 18, y1 - 10, max(1, int(h / 4))):
        d.line((x0 + 12, gy, x1 - 12, gy), fill=(*CYAN, 35), width=1)

    xs = np.linspace(x0 + 16, x1 - 16, 360)

    if mode == "amp":
        freqs = [0.020, 0.034, 0.051, 0.073]
        yy = np.zeros_like(xs, dtype=float)

        for k, f in enumerate(freqs):
            center = x0 + w * (0.22 + k * 0.17 + 0.01 * np.sin(phase + k))
            yy += (0.22 + 0.15 * np.sin(phase * 1.3 + k)) * np.exp(-0.5 * ((xs - center) / (5 + k * 1.5)) ** 2)

        yy += 0.05 * np.sin(xs * 0.11 + phase * 4)
        ybase = y1 - 28
        scale = h * 0.72
    else:
        yy = (
            0.35 * np.sin(xs * 0.09 + phase * 2.0)
            + 0.18 * np.sin(xs * 0.27 - phase * 1.6)
            + 0.09 * rng.normal(0, 1, len(xs))
        )
        ybase = y0 + h / 2
        scale = h * 0.22

    pts = [(x, ybase - v * scale) for x, v in zip(xs, yy)]

    for p1, p2 in zip(pts[:-1], pts[1:]):
        d.line([p1, p2], fill=(*CYAN2, 235), width=max(1, int(2 * SX)))


def draw_live_text_and_bars(img, phase, i):
    d = ImageDraw.Draw(img)

    # LIVE dot
    lx, ly = LIVE_DOT
    live_alpha = int(100 + 155 * np.sin(phase * 5) ** 2)
    d.ellipse([lx - 8, ly - 8, lx + 8, ly + 8], fill=(*GREEN, live_alpha))

    # time overwrite
    tx, ty = TIME_POS
    d.rectangle([tx - 80, ty - 18, tx + 155, ty + 24], fill=(2, 7, 13, 190))
    seconds = int((i / FPS) * 12) % 86400
    hh = 12 + seconds // 3600
    mm = (seconds // 60) % 60
    ss = seconds % 60
    d.text((tx - 50, ty - 12), f"T+ {hh:02d}:{mm:02d}:{ss:02d}", font=FONT_SMALL, fill=(*CYAN2, 230))

    # bottom data buffer bar
    x0, y0, x1, y1 = map(int, BAR_BOX)
    d.rectangle([x0, y0, x1, y1], fill=(2, 7, 13, 160))
    d.rectangle([x0, y0, x1, y1], outline=(*CYAN, 110), width=1)

    fill = 0.22 + 0.74 * ((i % FRAMES) / FRAMES)

    for k in range(42):
        px = x0 + 4 + k * ((x1 - x0 - 8) / 42)
        pw = ((x1 - x0 - 8) / 42) - 2
        active = k / 42 < fill
        col = GREEN if active else CYAN
        a = int(170 if active else 45)
        d.rectangle([px, y0 + 3, px + pw, y1 - 3], fill=(*col, a))

    # changing numeric values in right panel
    values = [
        (sx(1162), sy(122), f"{0.593 + 0.002*np.sin(phase):.3f}"),
        (sx(1162), sy(214), f"{11480 + int(25*np.sin(phase*1.5)):05d}"),
        (sx(1162), sy(764), f"{92 + 3*np.sin(phase):04.1f}%"),
        (sx(1162), sy(802), f"{7 + 2*np.sin(phase*2):03.0f}%"),
    ]

    for x, y, text in values:
        d.rectangle([x - 8, y - 7, x + 110, y + 18], fill=(2, 7, 13, 185))
        d.text((x, y - 5), text, font=FONT_TINY, fill=(*CYAN2, 230))


def render_frame(i):
    phase = 2 * np.pi * i / FRAMES

    img = bg0.copy()

    draw_live_star(img, phase)
    draw_panel_graph(img, AMP_BOX, AMP_BG, phase, "amp")
    draw_panel_graph(img, RES_BOX, RES_BG, phase, "res")
    draw_live_text_and_bars(img, phase, i)

    # финальный glow только для наложенных контрастов
    final = img.convert("RGBA")
    return final.convert("RGB")


frames = [render_frame(i) for i in range(FRAMES)]

frames[0].save(
    OUT,
    save_all=True,
    append_images=frames[1:],
    duration=int(1000 / FPS),
    loop=0,
    disposal=2,
)

print(f"Saved: {OUT}")

Saved: animations/white_dwarf/white_dwarf_seismology_live.gif


In [11]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont, ImageEnhance

OUT_DIR = Path("media-site/animations/white_dwarf")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Подставь свой файл, если имя другое
BACKGROUND_PATH = Path("media-site/assets/WD3.jpg")

OUT = OUT_DIR / "white_dwarf_seismology_live.gif"

FPS = 24
FRAMES = 160

CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
ORANGE = (255, 160, 70)
RED = (255, 80, 110)
WHITE = (245, 250, 255)

rng = np.random.default_rng(26051)


def load_font(size):
    for path in [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()


FONT_SMALL = load_font(22)
FONT_TINY = load_font(16)


bg0 = Image.open(BACKGROUND_PATH).convert("RGBA")
W, H = bg0.size

# немного приглушаем исходник, чтобы живые слои читались
bg0 = ImageEnhance.Brightness(bg0).enhance(0.82)
bg0 = ImageEnhance.Contrast(bg0).enhance(1.08)

# Координаты под сгенерированный 1792x1024 растр.
# Если используешь другой размер, всё масштабируется автоматически.
SX = W / 1792
SY = H / 1024

def sx(x): return x * SX
def sy(y): return y * SY

STAR_CX = sx(500)
STAR_CY = sy(440)
STAR_R = sx(314)

AMP_BOX = [sx(1400), sy(380), sx(1715), sy(560)]
RES_BOX = [sx(1040), sy(600), sx(1405), sy(710)]
BAR_BOX = [sx(1240), sy(880), sx(1590), sy(905)]
LIVE_DOT = (sx(1572), sy(34))
TIME_POS = (sx(454), sy(833))


def make_star_texture(size=720):
    tex = Image.new("RGBA", (size, size), (0, 0, 0, 0))
    d = ImageDraw.Draw(tex)

    cx = cy = size / 2
    r = size * 0.47

    # базовый диск
    for rr in np.linspace(r, 2, 80):
        a = int(8 + 105 * (1 - rr / r) ** 1.7)
        d.ellipse([cx - rr, cy - rr, cx + rr, cy + rr], fill=(*WHITE, a))

    # волокна/трещины
    for _ in range(420):
        a0 = rng.uniform(0, 2 * np.pi)
        rr = rng.uniform(0.05, 0.95) * r
        x = cx + np.cos(a0) * rr
        y = cy + np.sin(a0) * rr

        length = rng.uniform(18, 75)
        a1 = a0 + rng.normal(0, 0.9)

        x2 = x + np.cos(a1) * length
        y2 = y + np.sin(a1) * length

        if (x2 - cx) ** 2 + (y2 - cy) ** 2 < r ** 2:
            alpha = int(rng.uniform(25, 95))
            d.line([x, y, x2, y2], fill=(*CYAN2, alpha), width=1)

    # яркие узлы
    for _ in range(42):
        a0 = rng.uniform(0, 2 * np.pi)
        rr = rng.uniform(0.05, 0.92) * r
        x = cx + np.cos(a0) * rr
        y = cy + np.sin(a0) * rr
        rad = rng.uniform(3, 10)
        d.ellipse([x - rad, y - rad, x + rad, y + rad], fill=(*WHITE, int(rng.uniform(120, 230))))

    mask = Image.new("L", (size, size), 0)
    md = ImageDraw.Draw(mask)
    md.ellipse([cx - r, cy - r, cx + r, cy + r], fill=255)

    tex.putalpha(mask)
    return tex


STAR_TEX = make_star_texture()


def crop_panel(base, box):
    x0, y0, x1, y1 = map(int, box)
    patch = base.crop((x0, y0, x1, y1)).convert("RGBA")
    patch = ImageEnhance.Brightness(patch).enhance(0.55)
    return patch


AMP_BG = crop_panel(bg0, AMP_BOX)
RES_BG = crop_panel(bg0, RES_BOX)


def draw_live_star(img, phase):
    layer = Image.new("RGBA", (W, H), (0, 0, 0, 0))

    rot = STAR_TEX.rotate(
        phase * 360 / (2 * np.pi) * 0.18,
        resample=Image.Resampling.BICUBIC,
    )

    size = int(STAR_R * 2.08)
    rot = rot.resize((size, size), Image.Resampling.LANCZOS)

    x = int(STAR_CX - size / 2)
    y = int(STAR_CY - size / 2)

    pulse = 0.82 + 0.18 * np.sin(phase * 2.4)

    alpha = rot.getchannel("A").point(lambda p: int(p * 0.42 * pulse))
    rot.putalpha(alpha)

    layer.alpha_composite(rot, (x, y))

    d = ImageDraw.Draw(layer)

    # seismic rings
    for k in range(5):
        rr = STAR_R * (0.28 + ((phase * 0.12 + k * 0.19) % 1.0))
        a = int(90 * (1 - rr / STAR_R))
        d.ellipse(
            [STAR_CX - rr, STAR_CY - rr, STAR_CX + rr, STAR_CY + rr],
            outline=(*CYAN2, max(0, a)),
            width=max(1, int(2 * SX)),
        )

    # rim pulse
    rim_a = int(115 + 80 * np.sin(phase * 5) ** 2)
    d.ellipse(
        [STAR_CX - STAR_R, STAR_CY - STAR_R, STAR_CX + STAR_R, STAR_CY + STAR_R],
        outline=(*WHITE, rim_a),
        width=max(2, int(3 * SX)),
    )

    glow = layer.filter(ImageFilter.GaussianBlur(5))
    img.alpha_composite(glow)
    img.alpha_composite(layer)


def draw_panel_graph(img, box, bg_patch, phase, mode="amp"):
    x0, y0, x1, y1 = map(int, box)
    img.alpha_composite(bg_patch, (x0, y0))

    d = ImageDraw.Draw(img)

    w = x1 - x0
    h = y1 - y0

    # сетка поверх затемнения
    for gx in range(x0 + 20, x1 - 10, max(1, int(w / 6))):
        d.line((gx, y0 + 14, gx, y1 - 14), fill=(*CYAN, 45), width=1)

    for gy in range(y0 + 18, y1 - 10, max(1, int(h / 4))):
        d.line((x0 + 12, gy, x1 - 12, gy), fill=(*CYAN, 35), width=1)

    xs = np.linspace(x0 + 16, x1 - 16, 360)

    if mode == "amp":
        freqs = [0.020, 0.034, 0.051, 0.073]
        yy = np.zeros_like(xs, dtype=float)

        for k, f in enumerate(freqs):
            center = x0 + w * (0.22 + k * 0.17 + 0.01 * np.sin(phase + k))
            yy += (0.22 + 0.15 * np.sin(phase * 1.3 + k)) * np.exp(-0.5 * ((xs - center) / (5 + k * 1.5)) ** 2)

        yy += 0.05 * np.sin(xs * 0.11 + phase * 4)
        ybase = y1 - 28
        scale = h * 0.72
    else:
        yy = (
            0.35 * np.sin(xs * 0.09 + phase * 2.0)
            + 0.18 * np.sin(xs * 0.27 - phase * 1.6)
            + 0.09 * rng.normal(0, 1, len(xs))
        )
        ybase = y0 + h / 2
        scale = h * 0.22

    pts = [(x, ybase - v * scale) for x, v in zip(xs, yy)]

    for p1, p2 in zip(pts[:-1], pts[1:]):
        d.line([p1, p2], fill=(*CYAN2, 235), width=max(1, int(2 * SX)))


def draw_live_text_and_bars(img, phase, i):
    d = ImageDraw.Draw(img)

    # LIVE dot
    lx, ly = LIVE_DOT
    live_alpha = int(100 + 155 * np.sin(phase * 5) ** 2)
    d.ellipse([lx - 8, ly - 8, lx + 8, ly + 8], fill=(*GREEN, live_alpha))

    # time overwrite
    tx, ty = TIME_POS
    d.rectangle([tx - 80, ty - 18, tx + 155, ty + 24], fill=(2, 7, 13, 190))
    seconds = int((i / FPS) * 12) % 86400
    hh = 12 + seconds // 3600
    mm = (seconds // 60) % 60
    ss = seconds % 60
    d.text((tx - 50, ty - 12), f"T+ {hh:02d}:{mm:02d}:{ss:02d}", font=FONT_SMALL, fill=(*CYAN2, 230))

    # bottom data buffer bar
    x0, y0, x1, y1 = map(int, BAR_BOX)
    d.rectangle([x0, y0, x1, y1], fill=(2, 7, 13, 160))
    d.rectangle([x0, y0, x1, y1], outline=(*CYAN, 110), width=1)

    fill = 0.22 + 0.74 * ((i % FRAMES) / FRAMES)

    for k in range(42):
        px = x0 + 4 + k * ((x1 - x0 - 8) / 42)
        pw = ((x1 - x0 - 8) / 42) - 2
        active = k / 42 < fill
        col = GREEN if active else CYAN
        a = int(170 if active else 45)
        d.rectangle([px, y0 + 3, px + pw, y1 - 3], fill=(*col, a))

    # changing numeric values in right panel
    values = [
        (sx(1162), sy(122), f"{0.593 + 0.002*np.sin(phase):.3f}"),
        (sx(1162), sy(214), f"{11480 + int(25*np.sin(phase*1.5)):05d}"),
        (sx(1162), sy(764), f"{92 + 3*np.sin(phase):04.1f}%"),
        (sx(1162), sy(802), f"{7 + 2*np.sin(phase*2):03.0f}%"),
    ]

    for x, y, text in values:
        d.rectangle([x - 8, y - 7, x + 110, y + 18], fill=(2, 7, 13, 185))
        d.text((x, y - 5), text, font=FONT_TINY, fill=(*CYAN2, 230))


def render_frame(i):
    phase = 2 * np.pi * i / FRAMES

    img = bg0.copy()

    draw_live_star(img, phase)
    draw_panel_graph(img, AMP_BOX, AMP_BG, phase, "amp")
    draw_panel_graph(img, RES_BOX, RES_BG, phase, "res")
    draw_live_text_and_bars(img, phase, i)

    # финальный glow только для наложенных контрастов
    final = img.convert("RGBA")
    return final.convert("RGB")


import subprocess
import shutil

WEBM_OUT = OUT_DIR / "white_dwarf_seismology_live.webm"
FRAME_DIR = OUT_DIR / "frames_webm"
FRAME_DIR.mkdir(parents=True, exist_ok=True)

# 1) Рендерим PNG-кадры
for idx in range(FRAMES):
    frame = render_frame(idx)
    frame.save(FRAME_DIR / f"frame_{idx:04d}.png")

# 2) Проверяем ffmpeg
if shutil.which("ffmpeg") is None:
    raise RuntimeError(
        "ffmpeg не найден. Установи его: brew install ffmpeg"
    )

# 3) Кодируем WEBM VP9
cmd = [
    "ffmpeg", "-y",
    "-framerate", str(FPS),
    "-i", str(FRAME_DIR / "frame_%04d.png"),
    "-c:v", "libvpx-vp9",
    "-b:v", "0",
    "-crf", "34",
    "-pix_fmt", "yuv420p",
    "-row-mt", "1",
    str(WEBM_OUT),
]

subprocess.run(cmd, check=True)

shutil.rmtree(FRAME_DIR, ignore_errors=True)

print(f"Saved WEBM: {WEBM_OUT}")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved WEBM: animations/white_dwarf/white_dwarf_seismology_live.webm


[out#0/webm @ 0x120e12a70] video:1633KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.099793%
frame=  160 fps= 29 q=34.0 Lsize=    1635KiB time=00:00:06.66 bitrate=2009.0kbits/s speed=1.19x    


In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont, ImageEnhance

OUT_DIR = Path("media-site/animations/white_dwarf")
OUT_DIR.mkdir(parents=True, exist_ok=True)

BACKGROUND_PATH = Path("media-site/assets/WD3.jpg")
ZONES_PATH = Path("media-site/assets/zones.json")

WEBM_OUT = OUT_DIR / "white_dwarf_seismology_live.webm"
GIF_OUT = OUT_DIR / "white_dwarf_seismology_live.gif"

FPS = 24
FRAMES = 160
EXPORT_GIF = False
EXPORT_WEBM = True

CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
ORANGE = (255, 160, 70)
RED = (255, 80, 110)
WHITE = (245, 250, 255)

rng = np.random.default_rng(26051)


def load_font(size):
    for path in [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()


FONT_SMALL = load_font(22)
FONT_TINY = load_font(16)


bg0 = Image.open(BACKGROUND_PATH).convert("RGBA")
W, H = bg0.size

bg0 = ImageEnhance.Brightness(bg0).enhance(0.82)
bg0 = ImageEnhance.Contrast(bg0).enhance(1.08)

with open(ZONES_PATH, "r", encoding="utf-8") as f:
    ZONES = json.load(f)


def box_px(name):
    if name not in ZONES:
        raise KeyError(f"Zone '{name}' not found in {ZONES_PATH}")

    x0, y0, x1, y1 = ZONES[name]
    return [
        x0 * W,
        y0 * H,
        x1 * W,
        y1 * H,
    ]


def point_px(name):
    x0, y0, x1, y1 = box_px(name)
    return (
        (x0 + x1) / 2,
        (y0 + y1) / 2,
    )


def star_from_area(name="STAR_AREA"):
    x0, y0, x1, y1 = box_px(name)
    cx = (x0 + x1) / 2
    cy = (y0 + y1) / 2
    r = min(x1 - x0, y1 - y0) / 2
    return cx, cy, r


STAR_CX, STAR_CY, STAR_R = star_from_area("STAR_AREA")

AMP_BOX = box_px("AMP_BOX")
RES_BOX = box_px("RES_BOX")
BAR_BOX = box_px("BAR_BOX")
LIVE_DOT = point_px("LIVE_DOT")
TIME_POS = point_px("TIME_POS")

SX = W / 1792


def make_star_texture(size=720):
    tex = Image.new("RGBA", (size, size), (0, 0, 0, 0))
    d = ImageDraw.Draw(tex)

    cx = cy = size / 2
    r = size * 0.47

    for rr in np.linspace(r, 2, 80):
        a = int(8 + 105 * (1 - rr / r) ** 1.7)
        d.ellipse([cx - rr, cy - rr, cx + rr, cy + rr], fill=(*WHITE, a))

    for _ in range(420):
        a0 = rng.uniform(0, 2 * np.pi)
        rr = rng.uniform(0.05, 0.95) * r
        x = cx + np.cos(a0) * rr
        y = cy + np.sin(a0) * rr

        length = rng.uniform(18, 75)
        a1 = a0 + rng.normal(0, 0.9)

        x2 = x + np.cos(a1) * length
        y2 = y + np.sin(a1) * length

        if (x2 - cx) ** 2 + (y2 - cy) ** 2 < r ** 2:
            alpha = int(rng.uniform(25, 95))
            d.line([x, y, x2, y2], fill=(*CYAN2, alpha), width=1)

    for _ in range(42):
        a0 = rng.uniform(0, 2 * np.pi)
        rr = rng.uniform(0.05, 0.92) * r
        x = cx + np.cos(a0) * rr
        y = cy + np.sin(a0) * rr
        rad = rng.uniform(3, 10)
        d.ellipse([x - rad, y - rad, x + rad, y + rad], fill=(*WHITE, int(rng.uniform(120, 230))))

    mask = Image.new("L", (size, size), 0)
    md = ImageDraw.Draw(mask)
    md.ellipse([cx - r, cy - r, cx + r, cy + r], fill=255)

    tex.putalpha(mask)
    return tex


STAR_TEX = make_star_texture()


def crop_panel(base, box):
    x0, y0, x1, y1 = map(int, box)
    patch = base.crop((x0, y0, x1, y1)).convert("RGBA")
    patch = ImageEnhance.Brightness(patch).enhance(0.55)
    return patch


AMP_BG = crop_panel(bg0, AMP_BOX)
RES_BG = crop_panel(bg0, RES_BOX)


def draw_live_star(img, phase):
    layer = Image.new("RGBA", (W, H), (0, 0, 0, 0))

    rot = STAR_TEX.rotate(
        phase * 360 / (2 * np.pi) * 0.18,
        resample=Image.Resampling.BICUBIC,
    )

    size = int(STAR_R * 2.08)
    rot = rot.resize((size, size), Image.Resampling.LANCZOS)

    x = int(STAR_CX - size / 2)
    y = int(STAR_CY - size / 2)

    pulse = 0.82 + 0.18 * np.sin(phase * 2.4)

    alpha = rot.getchannel("A").point(lambda p: int(p * 0.42 * pulse))
    rot.putalpha(alpha)

    layer.alpha_composite(rot, (x, y))
    d = ImageDraw.Draw(layer)

    for k in range(5):
        rr = STAR_R * (0.28 + ((phase * 0.12 + k * 0.19) % 1.0))
        a = int(90 * (1 - rr / STAR_R))
        d.ellipse(
            [STAR_CX - rr, STAR_CY - rr, STAR_CX + rr, STAR_CY + rr],
            outline=(*CYAN2, max(0, a)),
            width=max(1, int(2 * SX)),
        )

    rim_a = int(115 + 80 * np.sin(phase * 5) ** 2)
    d.ellipse(
        [STAR_CX - STAR_R, STAR_CY - STAR_R, STAR_CX + STAR_R, STAR_CY + STAR_R],
        outline=(*WHITE, rim_a),
        width=max(2, int(3 * SX)),
    )

    glow = layer.filter(ImageFilter.GaussianBlur(5))
    img.alpha_composite(glow)
    img.alpha_composite(layer)


def draw_panel_graph(img, box, bg_patch, phase, mode="amp"):
    x0, y0, x1, y1 = map(int, box)
    img.alpha_composite(bg_patch, (x0, y0))

    d = ImageDraw.Draw(img)

    w = x1 - x0
    h = y1 - y0

    for gx in range(x0 + 20, x1 - 10, max(1, int(w / 6))):
        d.line((gx, y0 + 14, gx, y1 - 14), fill=(*CYAN, 45), width=1)

    for gy in range(y0 + 18, y1 - 10, max(1, int(h / 4))):
        d.line((x0 + 12, gy, x1 - 12, gy), fill=(*CYAN, 35), width=1)

    xs = np.linspace(x0 + 16, x1 - 16, 360)

    if mode == "amp":
        yy = np.zeros_like(xs, dtype=float)

        for k in range(4):
            center = x0 + w * (0.22 + k * 0.17 + 0.01 * np.sin(phase + k))
            yy += (0.22 + 0.15 * np.sin(phase * 1.3 + k)) * np.exp(
                -0.5 * ((xs - center) / (5 + k * 1.5)) ** 2
            )

        yy += 0.05 * np.sin(xs * 0.11 + phase * 4)
        ybase = y1 - 28
        scale = h * 0.72

    else:
        yy = (
            0.35 * np.sin(xs * 0.09 + phase * 2.0)
            + 0.18 * np.sin(xs * 0.27 - phase * 1.6)
            + 0.09 * rng.normal(0, 1, len(xs))
        )
        ybase = y0 + h / 2
        scale = h * 0.22

    pts = [(x, ybase - v * scale) for x, v in zip(xs, yy)]

    for p1, p2 in zip(pts[:-1], pts[1:]):
        d.line([p1, p2], fill=(*CYAN2, 235), width=max(1, int(2 * SX)))


def draw_live_text_and_bars(img, phase, i):
    d = ImageDraw.Draw(img)

    lx, ly = LIVE_DOT
    live_alpha = int(100 + 155 * np.sin(phase * 5) ** 2)
    d.ellipse([lx - 8, ly - 8, lx + 8, ly + 8], fill=(*GREEN, live_alpha))

    tx, ty = TIME_POS
    d.rectangle([tx - 80, ty - 18, tx + 155, ty + 24], fill=(2, 7, 13, 190))

    seconds = int((i / FPS) * 12) % 86400
    hh = 12 + seconds // 3600
    mm = (seconds // 60) % 60
    ss = seconds % 60

    d.text((tx - 50, ty - 12), f"T+ {hh:02d}:{mm:02d}:{ss:02d}", font=FONT_SMALL, fill=(*CYAN2, 230))

    x0, y0, x1, y1 = map(int, BAR_BOX)
    d.rectangle([x0, y0, x1, y1], fill=(2, 7, 13, 160))
    d.rectangle([x0, y0, x1, y1], outline=(*CYAN, 110), width=1)

    fill = 0.22 + 0.74 * ((i % FRAMES) / FRAMES)

    for k in range(42):
        px = x0 + 4 + k * ((x1 - x0 - 8) / 42)
        pw = ((x1 - x0 - 8) / 42) - 2
        active = k / 42 < fill
        col = GREEN if active else CYAN
        a = int(170 if active else 45)
        d.rectangle([px, y0 + 3, px + pw, y1 - 3], fill=(*col, a))


def render_frame(i):
    phase = 2 * np.pi * i / FRAMES

    img = bg0.copy()

    draw_live_star(img, phase)
    draw_panel_graph(img, AMP_BOX, AMP_BG, phase, "amp")
    draw_panel_graph(img, RES_BOX, RES_BG, phase, "res")
    draw_live_text_and_bars(img, phase, i)

    return img.convert("RGB")


frames = [render_frame(i) for i in range(FRAMES)]

if EXPORT_GIF:
    frames[0].save(
        GIF_OUT,
        save_all=True,
        append_images=frames[1:],
        duration=int(1000 / FPS),
        loop=0,
        disposal=2,
    )
    print(f"Saved GIF: {GIF_OUT}")

if EXPORT_WEBM:
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg не найден. Установи: brew install ffmpeg")

    FRAME_DIR = OUT_DIR / "frames_webm"
    FRAME_DIR.mkdir(parents=True, exist_ok=True)

    for idx, frame in enumerate(frames):
        frame.save(FRAME_DIR / f"frame_{idx:04d}.png")

    cmd = [
        "ffmpeg", "-y",
        "-framerate", str(FPS),
        "-i", str(FRAME_DIR / "frame_%04d.png"),
        "-c:v", "libvpx-vp9",
        "-b:v", "0",
        "-crf", "34",
        "-pix_fmt", "yuv420p",
        "-row-mt", "1",
        str(WEBM_OUT),
    ]

    subprocess.run(cmd, check=True)
    shutil.rmtree(FRAME_DIR, ignore_errors=True)
    print(f"Saved WEBM: {WEBM_OUT}")